# Silver notebook layer for the data transformation and cleaning of ERP location data
Location: workspace.bronze.erp_loc_a101

1. Check data types correctness
2. Rename headers
3. Trim of strings
4. Customer ID normalization. Possible Country naming conventions, which can be normalized

## Read data and spark libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

df = spark.table("workspace.bronze.erp_loc_a101")
df.display()

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Customer cleanup and country normalization

In [0]:
df = df.withColumn("CID", F.regexp_replace(col("CID"), "-", ""))
df.limit(10).display()

In [0]:
# Country column
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
     .when(col("cntry").isin("US", "USA"), "United States")
     .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
     .otherwise(col("cntry"))
)

## Renaming columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Write the dataframe as table in the silver layer catalog

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_locations")